<a href="https://colab.research.google.com/github/Tamil-Ilakkiya1404/Canteen-Management-System/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import math

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
def load_df(fp):
	df= pd.read_csv(fp, names=['id', 'ip', 'label', 'text']).set_index('id')
	df.drop_duplicates(inplace=True)
	df.dropna(inplace=True)
	df.info()
	return df

In [ ]:
df_trn= load_df('twitter_training.csv')
print()
df_tst= load_df('twitter_validation.csv')

In [ ]:
df_trn.columns.tolist()

In [ ]:
df_tst.columns.tolist()

In [ ]:
df_trn.isnull().sum()

In [ ]:
df_tst.isnull().sum()

In [ ]:
df_trn.info()

In [ ]:
df_trn.describe()

In [ ]:
df_trn.sample(5)

In [ ]:
df_trn['label'].value_counts()

In [ ]:
df_trn= df_trn[df_trn['label']!='Irrelevant']
df_trn.info()

In [ ]:
nlp= spacy.load('en_core_web_sm')

In [ ]:
def process_text(s):
	out= []
	for token in nlp(s):
		if not token.is_stop and not token.is_punct:
			out.append(token.lemma_)
	return ' '.join(out)

In [ ]:
df_trn['fltr']= df_trn['text'].apply(process_text)

In [ ]:
!pip install pandarallel

In [ ]:
from pandarallel import pandarallel

# Initialize Pandarallel
pandarallel.initialize()

# Apply process_text in parallel
df_trn['fltr'] = df_trn['text'].parallel_apply(process_text)

In [ ]:
df_tst.sample(5)

In [ ]:
# Fit the LabelEncoder on the combined unique labels from both training and test sets
enc = LabelEncoder()
enc.fit(pd.concat([df_trn['label'], df_tst['label']], ignore_index=True).unique())

# Transform the labels in the training and test sets
y_trn = enc.transform(df_trn['label'])
y_tst = enc.transform(df_tst['label'])

In [ ]:
df_tst['text'].parallel_apply(process_text)
df_tst['fltr'] = df_tst['text'].parallel_apply(process_text)
vct= TfidfVectorizer()
X_trn= vct.fit_transform(df_trn['fltr'])
X_tst= vct.transform(df_tst['fltr'])

In [ ]:
svc_model = SVC(kernel='linear')

# Train the model
svc_model.fit(X_trn, y_trn)

# Evaluate the model
accuracy = svc_model.score(X_tst, y_tst)
print(f'Model Accuracy: {accuracy:.2f}')

In [ ]:
y_pred=svc_model.predict(X_tst)
trnScore= svc_model.score(X_trn, y_trn)
tstScore= svc_model.score(X_tst, y_tst)
cm= confusion_matrix(y_tst, y_pred)
cr= classification_report(y_tst, y_pred)

In [ ]:
print("Train Score:",trnScore)
print("Test Score:",tstScore)
print("Confusion Matrix:",cm)
print("Classification Report:",cr)

In [ ]:
#training data distribution
plt.figure(figsize=(8, 6))
sns.countplot(x=y_trn)
plt.title('Class Distribution in Training Data')
plt.xlabel('Classes')
plt.ylabel('Count')
plt.show()

# Testing data distribution
plt.figure(figsize=(8, 6))
sns.countplot(x=y_tst)
plt.title('Class Distribution in Testing Data')
plt.xlabel('Classes')
plt.ylabel('Count')
plt.show()

In [ ]:
import pickle

# Save the model
with open('svc_model.pkl', 'wb') as model_file:
    pickle.dump(svc_model, model_file)

# Save the vectorizer
with open('vectorizer.pkl', 'wb') as vectorizer_file:
    pickle.dump(vct, vectorizer_file)

print("Model and vectorizer saved successfully!")


In [ ]:
from google.colab import files

# Download the saved model and vectorizer files
files.download('svc_model.pkl')
files.download('vectorizer.pkl')
